# D1 · Parco fotovoltaico: dalla previsione meteo all'offerta di energia (PySharp)

Stessa analisi del notebook [C#](D1-fotovoltaico-csharp.ipynb), con la facciata `Interop` e `matplotlib`: **stessi numeri**, verificati da un test. Kernel JupyterNet `pysharp` su .NET 11.

## Il caso

Un **parco fotovoltaico da 1 MWp** (inverter da 600 kW AC) vende energia sul mercato del giorno prima. Il SCADA registra ogni ora irraggiamento, temperature, potenza AC e stato dell'inverter; un servizio meteo manda, il giorno prima, la **previsione di irraggiamento** per le 24 ore successive, con il suo errore.

**Chi lavora sui dati.** L'*energy manager* deve presentare l'offerta di energia di domani; chi fa *O&M* vuole sapere se l'impianto rende come dovrebbe (e se vale la pena lavare i moduli); la *proprietà* vuole sapere perché un giorno si è prodotto meno. La domanda del cliente (il trader): **«quanta energia offrite sul mercato domani?»**, e va data **con un'incertezza**, perché produrre meno di quanto offerto costa di più che produrre di più.

**Dati sintetici** (`generate-data.cs`, con seme): 8 settimane orarie. Le prime 7 per addestrare, l'ultima come prova mai vista. La verità nota è in `data/verita.json`: la pulizia dei moduli (giorno 30), un guasto dell'inverter (giorno 38, 5 ore) e l'errore del servizio meteo.

| Passo | Domanda | Strumento |
|---|---|---|
| 1 | L'impianto rende come dovrebbe? | KPI: performance ratio, disponibilità |
| 2 | Quanta potenza darà domani, **sapendo il meteo previsto**? | regressione sulla covariata, seasonal naive, MASE |
| 3 | Con quale incertezza? | quantili dei residui per ora, perdita pinball, copertura |
| 4 | E sull'energia dell'**intera giornata**? | traiettorie campionate (blocchi di residui) |
| 5 | Quanta energia offro? | quantile ottimo dato il costo dell'errore |
| 6 | KPI, dashboard, risposta al cliente | tile, grafici, testo con i numeri |

In [ ]:
import os, math, clr
import matplotlib.pyplot as plt

# La cartella di lavoro deve essere dentro il repository Numbers (scripts/run-notebooks.ps1 parte dalla radice).
root = os.getcwd()
while not os.path.exists(os.path.join(root, "src", "MathemaSharp.slnx")):
    parent = os.path.dirname(root)
    if parent == root:
        raise RuntimeError("Esegui il notebook con la cartella di lavoro dentro il repository MathemaSharp.")
    root = parent
clr.AddReference(os.path.join(root, "artifacts", "bin", "MathemaSharp.TimeSeries", "release", "MathemaSharp.TimeSeries.dll"))
from MathemaSharp.TimeSeries.Interop import Statistics, Metrics, Linear

def P(label, values):
    if isinstance(values, (int, float)):
        values = [values]
    print(label + ": " + " ".join("%.6f" % v for v in values))

data = os.path.join(root, "scenari", "D-fotovoltaico", "data")
f = open(os.path.join(data, "impianto_orario.csv"))
text = f.read()
f.close()
rows = [l.split(",") for l in text.split(chr(10))[1:] if l.strip() != ""]
G = [float(r[1]) for r in rows]
Gf = [float(r[2]) for r in rows]
Tair = [float(r[3]) for r in rows]
Pac = [float(r[5]) for r in rows]
ok = [float(r[6]) for r in rows]
TRAIN_DAYS = 49
TRAIN_HOURS = TRAIN_DAYS * 24
TOTAL = 56 * 24
P_MAX = 600.0   # kW AC massimi dell'inverter
P_NOM = 1000.0  # kWp installati

## 1 · Acquisire e misurare: l'impianto rende come dovrebbe?
Il **performance ratio** (IEC 61724) è l'energia prodotta divisa per quella che produrrebbe un impianto ideale con la stessa insolazione: $PR=E_{AC}/(H_{POA}\cdot P_{nom})$. Misura la salute dell'impianto *a prescindere dal meteo*. La **disponibilità** conta le ore di sole in cui l'inverter era in servizio.

In [ ]:
daily_energy = [0.0] * 56
daily_insolation = [0.0] * 56
pr = [float("nan")] * 56
for d in range(56):
    for h in range(24):
        daily_energy[d] += Pac[d * 24 + h]
        daily_insolation[d] += G[d * 24 + h] / 1000
    if daily_insolation[d] > 0.5:
        pr[d] = daily_energy[d] / (daily_insolation[d] * P_NOM)

def mean_pr(first, last):
    values = [pr[d] for d in range(first, last)]
    return sum(values) / len(values)

P("pr_prima_pulizia", mean_pr(24, 30))
P("pr_dopo_pulizia", mean_pr(31, 37))
P("pr_giorno_guasto", pr[38])
sunny = [i for i in range(TOTAL) if G[i] > 50]
P("disponibilita", sum([1 for i in sunny if ok[i] == 1]) / len(sunny))
P("ore_al_limite_inverter", sum([1 for p in Pac if p >= P_MAX - 0.05]))

Il PR sale dopo la pulizia dei moduli (giorno 30) e precipita nel giorno del guasto: il KPI vede entrambi. Ma la differenza dovuta alla pulizia è di pochi punti: serve una **media su più giorni** per distinguerla dal rumore.

In [ ]:
plt.figure(figsize=(8, 2.8))
plt.plot(range(56), pr)
plt.title("Performance ratio giornaliero (pulizia al giorno 30, guasto al 38)")
plt.show()

## 2 · Prevedere la potenza di domani, sapendo il meteo previsto
La potenza dipende dall'irraggiamento e un servizio meteo lo prevede il giorno prima: è la **covariata nota nel futuro** (con il suo errore). Modello: regressione $P\approx\beta_0+\beta_1G_{prev}+\beta_2G_{prev}\,T_{aria}$, limitata a 0…600 kW, addestrata a QR sulle ore in cui l'inverter era in servizio. Riferimento: il **seasonal naive a un giorno** (la stessa ora di ieri).

In [ ]:
train_rows = [i for i in range(TRAIN_HOURS) if ok[i] == 1]   # fuori le ore di guasto: non sono "normali"
flat = []
y = []
for i in train_rows:
    flat.extend([1.0, Gf[i], Gf[i] * Tair[i]])
    y.append(Pac[i])
beta = list(Linear.LeastSquares(flat, len(train_rows), 3, y))
P("beta", beta)

def predict(i):
    if Gf[i] <= 0:
        return 0.0
    return min(max(beta[0] + beta[1] * Gf[i] + beta[2] * Gf[i] * Tair[i], 0.0), P_MAX)

train = Pac[:TRAIN_HOURS]
test = Pac[TRAIN_HOURS:]
model = [predict(i) for i in range(TRAIN_HOURS, TRAIN_HOURS + 168)]
persistence = Pac[TRAIN_HOURS - 24:TOTAL - 24]   # ieri alla stessa ora
P("mase_modello", Metrics.Mase(test, model, train, 24))
P("mase_persistenza", Metrics.Mase(test, persistence, train, 24))
try:
    Metrics.Mape(test, model)
    P("mape_definita", 1)
except ValueError:
    P("mape_definita", 0)   # di notte la potenza e 0

**La MAPE non si può calcolare**: di notte la potenza vale zero e la libreria lo dichiara invece di restituire un numero senza senso. È il motivo per cui in questo libro si usa la **MASE**: definita anche con gli zeri e indipendente dalla scala.

## 3 · Con quale incertezza? Quantili dei residui per ora del giorno
Le previsioni sbagliano in modo diverso a ogni ora (a mezzogiorno l'errore è grande, di notte nullo). Si prendono i **residui in campione** del modello (potenza reale − prevista) **ora per ora** e se ne calcolano i quantili: sono gli scarti da sommare alla previsione per ottenere P10, P50, P90.

In [ ]:
residual = [Pac[i] - predict(i) for i in range(TRAIN_HOURS)]

def hour_quantile(hour, p):
    return Statistics.Quantile([residual[i] for i in train_rows if i % 24 == hour], p)

q10 = [hour_quantile(h, 0.1) for h in range(24)]
q50 = [hour_quantile(h, 0.5) for h in range(24)]
q90 = [hour_quantile(h, 0.9) for h in range(24)]

def band(i, q):
    if Gf[i] <= 0:
        return 0.0
    return min(max(predict(i) + q[i % 24], 0.0), P_MAX)

lo = [band(i, q10) for i in range(TRAIN_HOURS, TOTAL)]
mid = [band(i, q50) for i in range(TRAIN_HOURS, TOTAL)]
hi = [band(i, q90) for i in range(TRAIN_HOURS, TOTAL)]
P("pinball_p10", Metrics.PinballLoss(test, lo, 0.1))
P("pinball_p50", Metrics.PinballLoss(test, mid, 0.5))
P("pinball_p90", Metrics.PinballLoss(test, hi, 0.9))
P("copertura_p10_p90_tutte_le_ore", Metrics.IntervalCoverage(test, lo, hi))
# Di notte la potenza e 0 e l'intervallo e [0, 0]: contiene sempre il valore e gonfia la copertura. Si misura solo sulle ore di sole.
daylight = [t for t in range(168) if Gf[TRAIN_HOURS + t] > 0]
coverage_day = Metrics.IntervalCoverage([test[t] for t in daylight], [lo[t] for t in daylight], [hi[t] for t in daylight])
P("copertura_p10_p90_ore_di_sole", coverage_day)

L'intervallo P10–P90 dovrebbe contenere l'**80 %** delle osservazioni. Contando **tutte** le ore la copertura sembra alta, ma è un'illusione: di notte la potenza è zero e l'intervallo è [0, 0], quindi contiene sempre il valore. Misurata **solo sulle ore di sole** è il numero che conta; se risultasse più bassa dell'80 %, l'intervallo sarebbe troppo ottimista (i residui in campione sottostimano l'errore su dati nuovi).

In [ ]:
a = 96
b = 168
plt.figure(figsize=(8, 3))
plt.plot(range(b - a), test[a:b], label="reale")
plt.plot(range(b - a), mid[a:b], linestyle="--", label="P50")
plt.fill_between(range(b - a), lo[a:b], hi[a:b], alpha=0.25, label="P10-P90")
plt.title("Ultimi 3 giorni: potenza reale e fascia P10-P90 (kW, asse x: ore)")
plt.legend()
plt.show()

## 4 · E l'energia dell'**intera giornata**?
La domanda del trader riguarda i kWh del giorno, non la potenza di un'ora. **Sommare i P10 orari non dà il P10 del giorno**: equivale a supporre che tutte le 24 ore sbaglino insieme nel peggiore dei modi. Si preservano invece le **dipendenze nel tempo** campionando traiettorie: a ogni giornata di addestramento si prende il suo intero vettore di residui, lo si somma alla previsione e si ottiene un'energia possibile. Dalle 48 energie possibili si prendono i quantili (giorno del guasto escluso).

In [ ]:
ok_days = [d for d in range(TRAIN_DAYS) if d != 38]
p10_day = []
p25_day = []
p50_day = []
p90_day = []
actual_day = []
sum_lo = []
sum_hi = []
for d in range(7):
    totals = []
    for k in ok_days:
        s = 0.0
        for h in range(24):
            i = TRAIN_HOURS + d * 24 + h
            if Gf[i] > 0:
                s += min(max(model[d * 24 + h] + residual[k * 24 + h], 0.0), P_MAX)
        totals.append(s)
    p10_day.append(Statistics.Quantile(totals, 0.1))
    p25_day.append(Statistics.Quantile(totals, 0.25))
    p50_day.append(Statistics.Quantile(totals, 0.5))
    p90_day.append(Statistics.Quantile(totals, 0.9))
    a_sum = 0.0
    lo_sum = 0.0
    hi_sum = 0.0
    for h in range(24):
        a_sum += test[d * 24 + h]
        lo_sum += lo[d * 24 + h]
        hi_sum += hi[d * 24 + h]
    actual_day.append(a_sum)
    sum_lo.append(lo_sum)
    sum_hi.append(hi_sum)
P("energia_reale", actual_day)
P("energia_p10", p10_day)
P("energia_p50", p50_day)
P("energia_p90", p90_day)
P("energia_somma_p10", sum_lo)
P("energia_somma_p90", sum_hi)

def inside(low, high):
    return sum([1 for d in range(7) if actual_day[d] >= low[d] and actual_day[d] <= high[d]])

P("giorni_dentro_traiettorie", inside(p10_day, p90_day))
P("giorni_dentro_somma_oraria", inside(sum_lo, sum_hi))
P("larghezza_traiettorie_kwh", sum([p90_day[d] - p10_day[d] for d in range(7)]) / 7)
P("larghezza_somma_oraria_kwh", sum([sum_hi[d] - sum_lo[d] for d in range(7)]) / 7)

La banda **dalle traiettorie** è più stretta di quella ottenuta sommando i quantili orari (circa **1.5 MWh** contro **1.9 MWh** in media): tiene conto di quanto gli errori si compensano durante il giorno, mentre la somma assume il caso peggiore su tutte le ore insieme, ed è per questo che contiene sempre tutti e sette i giorni. I giorni dentro la banda delle traiettorie sono **4 su 7** contro l'80 % atteso: con soli 7 giorni è compatibile con la sorte (vedere 4 o meno su 7 con probabilità 0.8 capita circa 15 volte su 100). **Non è una calibrazione**: ci vogliono molte più origini di prova.

## 5 · Quanta energia offrire? Il quantile che costa meno
Sul mercato, **produrre meno di quanto offerto costa di più** che produrre di più: se lo scarto per difetto costa $a$ per kWh e quello per eccesso $b$, l'offerta che minimizza il costo atteso è il **quantile di livello $b/(a+b)$** della distribuzione dell'energia: è la perdita pinball del capitolo sulle previsioni probabilistiche. Con $a=3,\;b=1$ il livello è **0.25**.

In [ ]:
def cost(offer, actual):
    return 3 * max(offer - actual, 0) + 1 * max(actual - offer, 0)   # a = 3 (difetto), b = 1 (eccesso)

def week_cost(offers):
    return sum([cost(offers[d], actual_day[d]) for d in range(7)])

P("costo_offerta_p10", week_cost(p10_day))
P("costo_offerta_p25", week_cost(p25_day))
P("costo_offerta_p50", week_cost(p50_day))
P("costo_offerta_p90", week_cost(p90_day))
offers = [("p10", p10_day), ("p25", p25_day), ("p50", p50_day), ("p90", p90_day)]
best_offer = min(offers, key=lambda o: week_cost(o[1]))
print("offerta_migliore: " + best_offer[0])

Sulla settimana di prova il livello 0.25 costa meno di offrire la mediana (P50) e molto meno del P90, come la teoria dice. Con sette giorni è una conferma, non una dimostrazione: la teoria vale **in media**, e il costo di una settimana è rumoroso.

## 6 · Comunicare: KPI, dashboard e risposta al cliente

In [ ]:
STATUS = {"ok": "&#10003; nella norma", "warn": "&#9650; attenzione", "alarm": "&#10006; allarme", "": ""}

def tiles(items):
    html = '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:8px;font-family:system-ui,sans-serif">'
    for label, value, note, status in items:
        html += '<div style="background:#fcfcfb;border:1px solid #e4e3df;border-radius:8px;padding:10px 12px">'
        html += '<div style="font-size:12px;color:#52514e">' + label + '</div>'
        html += '<div style="font-size:26px;font-weight:600;color:#0b0b0b;line-height:1.2">' + value + '</div>'
        html += '<div style="font-size:12px;color:#52514e">' + note + '</div>'
        if STATUS[status] != "":
            html += '<div style="font-size:12px;font-weight:600;color:#0b0b0b;margin-top:2px">' + STATUS[status] + '</div>'
        html += '</div>'
    display_html(html + '</div>')

def quote(title, text):
    display_html('<div style="background:#fcfcfb;border-left:4px solid #2a78d6;padding:10px 14px;margin:8px 0;font-family:system-ui,sans-serif;color:#0b0b0b"><div style="font-size:12px;color:#52514e">' + title + '</div><div style="font-size:14px">' + text + '</div></div>')

last = 6
tiles([
    ("Performance ratio", "%.3f" % mean_pr(31, 37), "dopo la pulizia, contro %.3f prima" % mean_pr(24, 30), "ok"),
    ("Disponibilit&agrave;", "%.1f %%" % (100.0 * sum([1 for i in sunny if ok[i] == 1]) / len(sunny)), "ore di sole con inverter in servizio; 5 ore di guasto il giorno 38", "ok"),
    ("Energia domani (P25)", "%.0f kWh" % p25_day[last], "P10 %.0f &ndash; P90 %.0f kWh; mediana %.0f" % (p10_day[last], p90_day[last], p50_day[last]), "ok"),
    ("Copertura P10&ndash;P90", "%.0f %%" % (100 * coverage_day), "ore di sole, sulla settimana di prova (atteso 80 %)", "warn" if coverage_day < 0.7 else "ok"),
    ("MASE del modello", "%.2f" % Metrics.Mase(test, model, train, 24), "contro %.2f della persistenza (ieri = oggi)" % Metrics.Mase(test, persistence, train, 24), "ok"),
])

In [ ]:
answer = ("Per il 20 settembre offriamo %.0f kWh. E il livello 25 %% della distribuzione dell'energia attesa (mediana %.0f kWh, intervallo 10-90 %% da %.0f a %.0f kWh): " % (p25_day[last], p50_day[last], p10_day[last], p90_day[last])
    + "il mercato penalizza il difetto tre volte l'eccesso, e offrire il livello 25 %% e cio che minimizza il costo atteso. "
    + "La previsione si basa sull'irraggiamento previsto dal servizio meteo: se il meteo cambia, aggiorneremo l'offerta entro la chiusura del mercato.")
quote("Risposta al cliente: quanta energia offrite sul mercato domani?", answer)

gain = 100 * (mean_pr(31, 37) - mean_pr(24, 30)) / mean_pr(31, 37)
oem = ("Il performance ratio medio e salito da %.3f a %.3f dopo la pulizia dei moduli del giorno 30: lo sporco costava circa il %.1f %% di energia. " % (mean_pr(24, 30), mean_pr(31, 37), gain)
    + "Il giorno del guasto (38) il PR e sceso a %.2f: 5 ore di inverter fermo." % pr[38])
quote("Nota per O&M", oem)

## Cosa abbiamo imparato
- **Il KPI giusto toglie il meteo**: il PR separa la salute dell'impianto dal tempo che fa.
- **Una covariata nota migliora molto** la previsione, ma porta con sé il suo errore: l'incertezza della previsione meteo *è* l'incertezza dell'offerta.
- **MAPE no, MASE sì**: con gli zeri notturni la MAPE non esiste.
- **I quantili orari non si sommano**: per l'energia del giorno si campionano traiettorie, preservando le dipendenze nel tempo.
- **Il costo dell'errore decide il quantile**: offrire la mediana non è ottimo quando il difetto costa più dell'eccesso.
- **Una settimana di prova non basta** per calibrare l'intervallo: la validazione su più origini (*rolling origin*) e la previsione conforme sono nel libro, non ancora nella libreria.